In [2]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

In [3]:
df_diamonds = pd.read_csv('./data/diamond_red.csv')
df_diamonds.head()

,carat,price
0,0.17,355
1,0.16,328
2,0.17,350
3,0.18,325
4,0.25,642


In [ ]:
sns.regplot(
    data = df_diamonds,
    x = 'carat',
    y = 'price'
);

## Univariate Model using sklearn

In [4]:
unimodel_sklearn = LinearRegression().fit(
    X = df_diamonds[['carat']].to_numpy(),
    y = df_diamonds['price'].to_numpy()
)

In [ ]:
df_diamonds['carat'].min()

In [ ]:
df_diamonds['carat'].max()

In [29]:
x_samples = np.linspace(
    df_diamonds['carat'].min(),
    df_diamonds['carat'].max(),
    10
)

In [30]:
unimodel_sklearn.predict(x_samples.reshape(-1,1))

array([ 186.89707499,  281.98993231,  377.08278963,  472.17564695,
        567.26850426,  662.36136158,  757.4542189 ,  852.54707622,
        947.63993353, 1042.73279085])

In [ ]:
sns.regplot(
    data = df_diamonds,
    x = 'carat',
    y = 'price'
);

sns.scatterplot(
    x = x_samples,
    y = unimodel_sklearn.predict(x_samples.reshape(-1,1)),
    color = 'red',
    marker = 'd'
);

## Prediction and Confidence Intervals



$$
\text{ci} = \hat{y_{0}} \pm t_{\alpha/2,n-2} \cdot s \sqrt{\frac{1}{n} + \frac{(x_{0} - \bar{x})^{2}}{\sum{(x_{i} - \bar{x})^{2}}}}
$$

$$
\text{pi} = \hat{y_{0}} \pm t_{\alpha/2,n-2} \cdot s \sqrt{1 + \frac{1}{n} + \frac{(x_{0} - \bar{x})^{2}}{\sum{(x_{i} - \bar{x})^{2}}}}
$$


In [31]:
x_0 = x_samples.reshape(-1,1)
yhat_0 = unimodel_sklearn.predict(x_0)
yhat_0

array([ 186.89707499,  281.98993231,  377.08278963,  472.17564695,
        567.26850426,  662.36136158,  757.4542189 ,  852.54707622,
        947.63993353, 1042.73279085])

In [32]:
y_hat = unimodel_sklearn.predict(df_diamonds[['carat']].to_numpy())
y = df_diamonds['price'].to_numpy()
n = df_diamonds['price'].shape[0]

# residual standard error
rse = np.sqrt(((y-y_hat)**2).sum()/(n-2))
rse

31.840522265031762

In [ ]:
# Total x variability squared
x_mean = df_diamonds[['carat']].to_numpy().mean()

ssx = ((df_diamonds['carat'] - x_mean)**2).sum()
rse*np.sqrt(1 + 1/n + ((x_0-x_mean)**2)/ssx)

# prediction intervals with alpha = 0.05
prediction_intervals = yhat_0.reshape(-1,1) + np.array([-1,1]) * stats.t.ppf(0.975, df = df_diamonds.shape[0]-2) * rse * np.sqrt(1 + 1/n + ((x_0-x_mean)**2)/ssx)
confidence_intervals = yhat_0.reshape(-1,1) + np.array([-1,1]) * stats.t.ppf(0.975, df = df_diamonds.shape[0]-2) * rse * np.sqrt(1/n + ((x_0-x_mean)**2)/ssx)

In [ ]:
df_predictions = pd.DataFrame(
    {
        'x_new' : x_samples,
        'y_new' : yhat_0,
        'mean_ci_lower' : confidence_intervals[:,0],
        'mean_ci_upper' : confidence_intervals[:,1],
        'mean_obs_lower' : prediction_intervals[:, 0],
        'mean_obs_upper' : prediction_intervals[:, 1],
    }
)

#prediction_intervals

In [ ]:
sns.regplot(
    data = df_diamonds,
    x = 'carat',
    y = 'price',
    scatter=True
);

sns.scatterplot(
    data = df_predictions,
    x = 'x_new',
    y = 'y_new',
    color = 'red',
    marker = 'd'
)

plt.vlines(
    x = df_predictions['x_new'].values,
    ymin = df_predictions['mean_ci_lower'].values,
    ymax = df_predictions['mean_ci_upper'].values
)

plt.vlines(
    x = df_predictions['x_new'].values + 0.001,
    ymin = df_predictions['mean_obs_lower'].values,
    ymax = df_predictions['mean_obs_upper'].values,
    color = 'red'
)

In [ ]:
df_predictions.head()

In [37]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from scipy import stats

def get_prediction(model, data, features, target, x_new, alpha = 0.05):
    """
    model: sklearn linear regression model
    data: dataframe
    feauter: list of strings
    target: string
    x_new: numpy array of new points
    alpha: confidence level
    """
    x_0 = x_new.reshape(-1,1)
    yhat_0 = model.predict(x_0)

    y_hat = unimodel_sklearn.predict(data[features].to_numpy())
    y = df_diamonds[target].to_numpy()
    n = df_diamonds[target].shape[0]

    # residual standard error
    rse = np.sqrt(((y-y_hat)**2).sum()/(n-2))
    
    # keeping consistency in transforming data contained in dataframe to matrix

    x_mean = data[features].to_numpy().mean()
    # x Total variability squared
    ssx = ((data[features] - x_mean)**2).sum().values
#     rse*np.sqrt(1 + 1/n + ((x_0-x_mean)**2)/ssx)

    print(ssx)
    prediction_intervals = yhat_0.reshape(-1,1) + np.array([-1,1]) * stats.t.ppf(0.975, df = n-2) * rse * np.sqrt(1 + 1/n + ((x_0-x_mean)**2)/ssx)
    confidence_intervals = yhat_0.reshape(-1,1) + np.array([-1,1]) * stats.t.ppf(0.975, df = n-2) * rse * np.sqrt(1/n + ((x_0-x_mean)**2)/ssx)
    
    df_predictions = pd.DataFrame(
        {
            'x_new' : x_samples,
            'y_new' : yhat_0,
            'mean_ci_lower' : confidence_intervals[:,0],
            'mean_ci_upper' : confidence_intervals[:,1],
            'mean_obs_lower' : prediction_intervals[:, 0],
            'mean_obs_upper' : prediction_intervals[:, 1],
        }
    )
    return(df_predictions)
    

In [38]:
x_samples = np.linspace(
    df_diamonds['carat'].min(),
    df_diamonds['carat'].max(),
    10
)

get_prediction(model = unimodel_sklearn, 
               data = df_diamonds, 
               features = ['carat'], 
               target = 'price', 
               x_new = x_samples, 
               alpha = 0.05)

[0.15156667]


,x_new,y_new,mean_ci_lower,mean_ci_upper,mean_obs_lower,mean_obs_upper
0,0.120000,186.897075,170.236695,203.557455,120.675421,253.118729
1,0.145556,281.989932,268.622812,295.357053,216.519182,347.460683
2,0.171111,377.082790,366.350069,387.815510,312.098711,442.066869
3,0.196667,472.175647,462.842781,481.508513,407.408050,536.943244
4,0.222222,567.268504,557.551859,576.985149,502.444493,632.092515
5,0.247778,662.361362,650.651378,674.071345,597.208749,727.513974
6,0.273333,757.454219,742.783364,772.125073,691.704898,823.203540
7,0.298889,852.547076,834.415779,870.678373,785.940146,919.154007
8,0.324444,947.639934,925.784645,969.495222,879.924405,1015.355462
9,0.350000,1042.732791,1017.004149,1068.461433,973.669760,1111.795822


In [7]:
df_diamonds

,carat,price
0,0.17,355
1,0.16,328
2,0.17,350
3,0.18,325
4,0.25,642
5,0.16,342
6,0.15,322
7,0.19,485
8,0.21,483
9,0.15,323
